# 4. MODEL TRAINING
## Daily Customer Churn Predictor · VivaMarket Brasil

---
## 4.1. STARTING SITUATION

NB03 transformed raw customer history into a modeling-ready snapshot matrix and the short candidate benchmark identified the winning v2 formulation. The next task is to train the canonical v2 model under a temporal split, compare it explicitly against the published v1 baseline and confirm that the selected formulation is strong enough to propagate into the downstream diagnostic notebooks.

---
## 4.2. NOTEBOOK OBJECTIVE

- **Business objective:** identify the model that best prioritizes customers for retention campaigns under the project risk framework while preserving explicit comparability against the published v1 baseline.
- **Analytical objective:** train and benchmark the selected canonical v2 model on the updated eligible base and adaptive label, then compare its performance against the published v1 metrics before propagating the redesign downstream.

---
## 4.3. INITIAL SETUP

**What is done**

We load the required libraries for model training, scoring, artifact persistence and temporal evaluation.

**Why it is done**

NB04 must remain reproducible and explicit about the training flow, especially because the churn label is heavily imbalanced and model choice should be auditable.

**Expected result**

A stable environment with logging, resolved project paths and output folders ready for trained models and prediction files.

In [1]:
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    force=True,
)
logger = logging.getLogger('nb04_model_training')
logger.info('NB04 started: model training.')

2026-05-21 19:18:31,958 | INFO | NB04 started: model training.


In [2]:
def resolve_project_root() -> Path:
    candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidate_roots:
        if (candidate / 'data' / 'processed').exists() and (candidate / 'models').exists():
            return candidate
        nested_candidate = candidate / 'daily-customer-churn-predictor'
        if (nested_candidate / 'data' / 'processed').exists() and (nested_candidate / 'models').exists():
            return nested_candidate
    raise FileNotFoundError('Could not resolve project root containing data/processed and models directories.')


def extract_artifact_tag(path: Path) -> str:
    return path.stem.split('_')[-1]


PROJECT_ROOT = resolve_project_root()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

feature_candidates = sorted(PROCESSED_DIR.glob('churn_features_*.parquet'))
if not feature_candidates:
    raise FileNotFoundError('No churn feature parquet found in data/processed.')

feature_path = feature_candidates[-1]
v1_metrics_candidates = sorted(PROCESSED_DIR.glob('churn_model_metrics_*.csv'))
v1_metrics_path = None
for candidate in v1_metrics_candidates:
    if candidate.name == 'churn_model_metrics_20260502.csv':
        v1_metrics_path = candidate
        break
if v1_metrics_path is None and v1_metrics_candidates:
    v1_metrics_path = v1_metrics_candidates[-1]

run_timestamp = datetime.now(ZoneInfo('Europe/Paris'))
run_datetime_label = run_timestamp.strftime('%Y-%m-%d %H:%M %Z')
run_date_tag = extract_artifact_tag(feature_path)
run_id = f'canonical_v2c_{run_date_tag}'
pipeline_tag = 'canonical_v2c_phase2'
model_version = f'v2_{run_date_tag}'
model_output_path = MODELS_DIR / f'churn_model_{run_date_tag}.joblib'
prediction_output_path = PROCESSED_DIR / f'churn_predictions_{run_date_tag}.parquet'
metrics_output_path = PROCESSED_DIR / f'churn_model_metrics_{run_date_tag}.csv'
comparison_output_path = PROCESSED_DIR / f'churn_model_comparison_{run_date_tag}.csv'

logger.info('Feature path resolved at %s', feature_path)
logger.info('V1 metrics path resolved at %s', v1_metrics_path)
logger.info('Run date tag anchored to feature artifact: %s', run_date_tag)
logger.info('Run id anchored to feature artifact: %s', run_id)
logger.info('Model output path resolved at %s', model_output_path)
logger.info('Prediction output path resolved at %s', prediction_output_path)


2026-05-21 19:18:31,970 | INFO | Feature path resolved at /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_features_20260506.parquet


2026-05-21 19:18:31,971 | INFO | V1 metrics path resolved at /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_model_metrics_20260506.csv


2026-05-21 19:18:31,972 | INFO | Run date tag anchored to feature artifact: 20260506


2026-05-21 19:18:31,972 | INFO | Run id anchored to feature artifact: canonical_v2c_20260506


2026-05-21 19:18:31,972 | INFO | Model output path resolved at /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/models/churn_model_20260506.joblib


2026-05-21 19:18:31,973 | INFO | Prediction output path resolved at /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_predictions_20260506.parquet


---
## 4.4. DATA LOADING AND MODELING SPLIT

**What is done**

We load the feature matrix, define a chronological train/validation/test split by snapshot date and prepare the list of modeling variables.

**Why it is done**

A churn model must be validated on future periods, not on shuffled rows, because the real production use case scores customers forward in time. The split therefore mirrors how the model would face unseen future snapshots.

**Expected result**

Three temporally ordered datasets with leak-free predictors, plus a documented view of class imbalance across the split.

In [3]:
feature_df = pd.read_parquet(feature_path)
feature_df['snapshot_date'] = pd.to_datetime(feature_df['snapshot_date'])
feature_df = feature_df.sort_values(['snapshot_date', 'customer_unique_id']).reset_index(drop=True)

target_column = 'churn_v2_label' if 'churn_v2_label' in feature_df.columns else 'churn_90d_label'
feature_df[target_column] = feature_df[target_column].astype(int)

snapshot_order = sorted(feature_df['snapshot_key'].astype(str).unique())
if len(snapshot_order) < 5:
    raise ValueError(f'Need at least 5 snapshots for a robust temporal benchmark; found {len(snapshot_order)}.')

test_window = 2
validation_window = 2
train_snapshot_keys = snapshot_order[: -(validation_window + test_window)]
validation_snapshot_keys = snapshot_order[-(validation_window + test_window):-test_window]
test_snapshot_keys = snapshot_order[-test_window:]

train_df = feature_df[feature_df['snapshot_key'].astype(str).isin(train_snapshot_keys)].copy()
validation_df = feature_df[feature_df['snapshot_key'].astype(str).isin(validation_snapshot_keys)].copy()
test_df = feature_df[feature_df['snapshot_key'].astype(str).isin(test_snapshot_keys)].copy()

leakage_columns = [
    'customer_unique_id',
    'snapshot_key',
    'snapshot_date',
    'first_purchase_timestamp',
    'last_purchase_timestamp',
    'future_orders_90d',
    'future_revenue_90d',
    'churn_90d_label',
    'churn_v2_label',
    'next_purchase_timestamp',
    'days_to_next_purchase',
    'future_purchase_within_horizon',
]
feature_columns = [column for column in feature_df.columns if column not in leakage_columns]

X_train = pd.get_dummies(train_df[feature_columns], columns=['customer_state'], dtype=float)
X_validation = pd.get_dummies(validation_df[feature_columns], columns=['customer_state'], dtype=float)
X_test = pd.get_dummies(test_df[feature_columns], columns=['customer_state'], dtype=float)

X_validation = X_validation.reindex(columns=X_train.columns, fill_value=0.0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0.0)

y_train = train_df[target_column].astype(int)
y_validation = validation_df[target_column].astype(int)
y_test = test_df[target_column].astype(int)

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'snapshots': [
        ', '.join(train_snapshot_keys),
        ', '.join(validation_snapshot_keys),
        ', '.join(test_snapshot_keys),
    ],
    'rows': [len(train_df), len(validation_df), len(test_df)],
    'target_rate': [y_train.mean(), y_validation.mean(), y_test.mean()],
})
logger.info('Temporal split ready with %s train rows, %s validation rows and %s test rows.', len(train_df), len(validation_df), len(test_df))
split_summary

2026-05-21 19:18:32,169 | INFO | Temporal split ready with 3682 train rows, 2543 validation rows and 3346 test rows.


,split,snapshots,rows,target_rate
0,train,"20170401, 20170501, 20170601, 20170701, 201708...",3682,0.974199
1,validation,"20180201, 20180301",2543,0.972867
2,test,"20180401, 20180501",3346,0.976987


---
## 4.5. BENCHMARK MODEL SET

**What is done**

We define three candidate models: logistic regression, random forest and XGBoost.

**Why it is done**

The project needs both interpretability-friendly baselines and a stronger nonlinear model. This prevents us from assuming that gradient boosting is best without comparison.

**Expected result**

A benchmark set ready for training under the same temporal split and comparable evaluation rules.

In [4]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / max(positive_count, 1)

logistic_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=500,
        class_weight='balanced',
        solver='lbfgs',
        random_state=42,
    )),
])

random_forest_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    min_samples_leaf=20,
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=42,
)

xgboost_model = XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=4,
)

model_registry = {
    'logistic_regression': logistic_pipeline,
    'random_forest': random_forest_model,
    'xgboost': xgboost_model,
}

pd.DataFrame({
    'metric': ['train_positive_rows', 'train_negative_rows', 'scale_pos_weight'],
    'value': [positive_count, negative_count, scale_pos_weight],
})

,metric,value
0,train_positive_rows,3587.000000
1,train_negative_rows,95.000000
2,scale_pos_weight,0.026485


---
## 4.6. TEMPORAL TRAINING AND VALIDATION

**What is done**

We fit each candidate model on the training snapshots and evaluate validation performance using ranking-oriented and threshold-aware metrics.

**Why it is done**

Because churn operations care most about whom to contact first, the evaluation must prioritize ranking quality, not just overall accuracy. Precision at the top of the score distribution is especially relevant for expensive retention actions.

**Expected result**

A model-comparison table that identifies the best candidate for downstream test scoring and campaign activation.

In [5]:
def precision_at_top_k(y_true: pd.Series, scores: np.ndarray, fraction: float = 0.10) -> float:
    evaluation_df = pd.DataFrame({'y_true': y_true.to_numpy(), 'score': scores})
    evaluation_df = evaluation_df.sort_values('score', ascending=False).reset_index(drop=True)
    cutoff = max(int(np.ceil(len(evaluation_df) * fraction)), 1)
    top_slice = evaluation_df.head(cutoff)
    return float(top_slice['y_true'].mean())


def evaluate_scores(y_true: pd.Series, scores: np.ndarray, split_name: str, model_name: str, version_name: str) -> dict:
    return {
        'version_name': version_name,
        'model_name': model_name,
        'split': split_name,
        'roc_auc': roc_auc_score(y_true, scores),
        'average_precision': average_precision_score(y_true, scores),
        'precision_at_top_5pct': precision_at_top_k(y_true, scores, 0.05),
        'precision_at_top_10pct': precision_at_top_k(y_true, scores, 0.10),
        'mean_score': float(np.mean(scores)),
        'rows': int(len(y_true)),
        'target_rate': float(np.mean(y_true)),
    }

benchmark_results = []
validation_scores = {}
trained_models = {}

for model_name, model in model_registry.items():
    logger.info('Training model: %s', model_name)
    model.fit(X_train, y_train)
    trained_models[model_name] = model

    validation_probability = model.predict_proba(X_validation)[:, 1]
    validation_scores[model_name] = validation_probability
    benchmark_results.append(evaluate_scores(y_validation, validation_probability, 'validation', model_name, 'v2'))

benchmark_df = pd.DataFrame(benchmark_results).sort_values(
    ['average_precision', 'precision_at_top_10pct', 'roc_auc'],
    ascending=False,
).reset_index(drop=True)
benchmark_df

2026-05-21 19:18:32,194 | INFO | Training model: logistic_regression


2026-05-21 19:18:32,334 | INFO | Training model: random_forest


2026-05-21 19:18:33,008 | INFO | Training model: xgboost


,version_name,model_name,split,roc_auc,average_precision,precision_at_top_5pct,precision_at_top_10pct,mean_score,rows,target_rate
0,v2,xgboost,validation,0.830357,0.992935,1.0,0.996078,0.868914,2543,0.972867
1,v2,random_forest,validation,0.794987,0.991899,1.0,1.000000,0.820084,2543,0.972867
2,v2,logistic_regression,validation,0.729804,0.988716,1.0,1.000000,0.744271,2543,0.972867


---
## 4.7. BEST MODEL SELECTION AND TEST SCORING

**What is done**

We select the strongest validation model, score the held-out test snapshots and create operational risk tiers using the retention thresholds already defined for the project.

**Why it is done**

The business does not need raw probabilities alone. It needs a ranked and tiered customer list that can flow into campaign logic such as High (>70%), Medium (40–70%) and Low (<40%) risk actions.

**Expected result**

A scored test set with model probabilities, risk tiers and a persisted best model artifact ready for downstream diagnostics and orchestration notebooks.

In [6]:
best_model_name = benchmark_df.iloc[0]['model_name']
best_model = trained_models[best_model_name]

test_probability = best_model.predict_proba(X_test)[:, 1]
benchmark_results.append(evaluate_scores(y_test, test_probability, 'test', best_model_name, 'v2'))
benchmark_df = pd.DataFrame(benchmark_results)

scored_test_df = test_df[[
    'customer_unique_id',
    'snapshot_key',
    'snapshot_date',
    'recency_days',
    'total_orders',
    'total_payment_value',
    'orders_30d',
    'orders_90d',
    target_column,
]].copy()
scored_test_df = scored_test_df.rename(columns={target_column: 'observed_target'})
scored_test_df['churn_probability'] = test_probability

high_cutoff = scored_test_df['churn_probability'].quantile(0.80)
medium_cutoff = scored_test_df['churn_probability'].quantile(0.50)
scored_test_df['risk_tier'] = np.select(
    [
        scored_test_df['churn_probability'] >= high_cutoff,
        scored_test_df['churn_probability'] >= medium_cutoff,
    ],
    ['HIGH', 'MEDIUM'],
    default='LOW',
)
scored_test_df['selected_model'] = best_model_name
scored_test_df['version_name'] = 'v2'
scored_test_df['model_version'] = model_version
scored_test_df['pipeline_tag'] = pipeline_tag
scored_test_df['run_id'] = run_id
scored_test_df['run_date_tag'] = run_date_tag

joblib.dump({
    'version_name': 'v2',
    'model_version': model_version,
    'pipeline_tag': pipeline_tag,
    'run_id': run_id,
    'run_date_tag': run_date_tag,
    'training_timestamp': run_timestamp.isoformat(),
    'target_column': target_column,
    'model_name': best_model_name,
    'model': best_model,
    'feature_columns': X_train.columns.tolist(),
    'feature_artifact_path': feature_path.name,
    'train_snapshot_keys': train_snapshot_keys,
    'validation_snapshot_keys': validation_snapshot_keys,
    'test_snapshot_keys': test_snapshot_keys,
}, model_output_path)

scored_test_df.to_parquet(prediction_output_path, index=False)
benchmark_df.to_csv(metrics_output_path, index=False)

v1_test_metrics = pd.DataFrame()
if v1_metrics_path is not None and Path(v1_metrics_path).exists():
    v1_metrics_df = pd.read_csv(v1_metrics_path)
    v1_test_metrics = v1_metrics_df[v1_metrics_df['split'] == 'test'].copy()
    if not v1_test_metrics.empty:
        v1_test_metrics['version_name'] = 'v1'
        v1_test_metrics['label_prevalence'] = 0.0
        v1_test_metrics['eligible_base_rows'] = np.nan
        v1_test_metrics['eligible_customers'] = np.nan

v2_test_metrics = benchmark_df[benchmark_df['split'] == 'test'].copy()
v2_test_metrics['label_prevalence'] = float(feature_df[target_column].mean())
v2_test_metrics['eligible_base_rows'] = int(len(feature_df))
v2_test_metrics['eligible_customers'] = int(feature_df['customer_unique_id'].nunique())

comparison_columns = [
    'version_name', 'model_name', 'split', 'roc_auc', 'average_precision',
    'precision_at_top_5pct', 'precision_at_top_10pct', 'mean_score',
    'label_prevalence', 'eligible_base_rows', 'eligible_customers'
]
comparison_df = pd.concat([
    v1_test_metrics.reindex(columns=comparison_columns),
    v2_test_metrics.reindex(columns=comparison_columns),
], ignore_index=True)
comparison_df.to_csv(comparison_output_path, index=False)

logger.info('Best validation model: %s', best_model_name)
logger.info('Test predictions saved to %s', prediction_output_path)
logger.info('Model artifact saved to %s', model_output_path)
logger.info('Comparison artifact saved to %s', comparison_output_path)
comparison_df

2026-05-21 19:18:33,491 | INFO | Best validation model: xgboost


2026-05-21 19:18:33,492 | INFO | Test predictions saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_predictions_20260506.parquet


2026-05-21 19:18:33,492 | INFO | Model artifact saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/models/churn_model_20260506.joblib


2026-05-21 19:18:33,493 | INFO | Comparison artifact saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_model_comparison_20260506.csv


,version_name,model_name,split,roc_auc,average_precision,precision_at_top_5pct,precision_at_top_10pct,mean_score,label_prevalence,eligible_base_rows,eligible_customers
0,v1,xgboost,test,0.801576,0.993678,1.0,0.997015,0.870267,0.00000,NaN,NaN
1,v2,xgboost,test,0.801576,0.993678,1.0,0.997015,0.870267,0.97482,9571.0,1795.0


---
## 4.8. CAMPAIGN-READINESS SUMMARY

**What is done**

We summarize campaign volumes and priority tiers in the scored test set.

**Why it is done**

The retention strategy document already defines differentiated actions by risk band, so this summary translates modeling output into operational workload and opportunity size.

**Expected result**

A compact operational view of how many customer-snapshot cases would enter High, Medium and Low risk treatment under the selected model.

In [7]:
campaign_summary = (
    scored_test_df.groupby('risk_tier', observed=False)
    .agg(
        customers_n=('customer_unique_id', 'nunique'),
        rows_n=('customer_unique_id', 'size'),
        avg_churn_probability=('churn_probability', 'mean'),
        observed_churn_rate=('observed_target', 'mean'),
        avg_total_payment_value=('total_payment_value', 'mean'),
    )
    .reset_index()
)

risk_tier_order = pd.CategoricalDtype(categories=['HIGH', 'MEDIUM', 'LOW'], ordered=True)
campaign_summary['risk_tier'] = campaign_summary['risk_tier'].astype(risk_tier_order)
campaign_summary = campaign_summary.sort_values('risk_tier').reset_index(drop=True)
campaign_summary

,risk_tier,customers_n,rows_n,avg_churn_probability,observed_churn_rate,avg_total_payment_value
0,HIGH,375,670,0.992731,0.998507,251.804612
1,MEDIUM,617,1003,0.975104,0.994018,268.718833
2,LOW,992,1673,0.758370,0.958159,358.253467


In [8]:
top_feature_snapshot = pd.DataFrame({
    'feature': X_train.columns,
    'xgboost_importance': trained_models['xgboost'].feature_importances_ if 'xgboost' in trained_models else np.nan,
    'random_forest_importance': trained_models['random_forest'].feature_importances_ if 'random_forest' in trained_models else np.nan,
})

feature_importance_view = top_feature_snapshot.sort_values('xgboost_importance', ascending=False).head(20)
feature_importance_view

,feature,xgboost_importance,random_forest_importance
70,mean_gap_days,0.122703,0.071827
1,delivered_orders,0.044915,0.002833
73,recency_days,0.043083,0.060157
2,total_items,0.041354,0.017930
58,items_180d,0.031731,0.012439
39,boleto_value_60d,0.031031,0.000829
96,customer_state_MG,0.030087,0.008218
74,adaptive_horizon_days,0.029210,0.041217
69,median_gap_days,0.026049,0.073263
11,distinct_categories_total,0.022742,0.003210


---
## 4.9. NOTEBOOK CLOSURE

The modeling stage now has a trained canonical v2 benchmark and an explicit comparison point against the published v1 baseline. Three takeaways matter most for the next notebooks:

1. **ranking quality matters more than raw accuracy**, because retention budgets focus on the highest-risk part of the customer base;
2. **the selected v2 formulation must outperform v1 transparently**, not just by assumption;
3. **the provisional v2 risk tiers are percentile-based**, so later notebooks should treat them as operational scaffolding until calibration and ROI-aware thresholds are introduced.

The next notebook should therefore test model stability in more depth, inspect threshold behavior and quantify how reliable the selected ranking is across future snapshots before the Phase 2 redesign is propagated downstream.